In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

In [ ]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import os

json_files = []
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        if filename.lower().endswith(".json"):
            json_files.append(os.path.join(dirname, filename))

print("JSON files found:")
for p in json_files:
    print(p)

In [ ]:
import pandas as pd

JSON_PATH = "/kaggle/input/datasets/bharatgangolli/icecat/icecat_data_train (1).json"

df = pd.read_json(JSON_PATH)

print("Shape:", df.shape)
print("Columns count:", len(df.columns))

In [ ]:
pd.set_option("display.max_columns", 200)

for i, col in enumerate(df.columns, 1):
    print(f"{i:02d}. {col}")

In [ ]:
possible_taxonomy_cols = [
    "pathlist_names",
    "pathlist_ids",
    "Category.Name.Value",
    "Category.CategoryID",
    "VirtualCategory"
]

for col in possible_taxonomy_cols:
    if col in df.columns:
        print(f"\n--- {col} ---")
        display(df[[col]].head(5))
    else:
        print(f"{col} -> NOT FOUND")

In [ ]:
def split_3levels(x):
    if pd.isna(x):
        return None, None, None
    parts = [p.strip() for p in str(x).split(">")]
    parts += [None] * (3 - len(parts))
    return parts[0], parts[1], parts[2]

df["L1"], df["L2"], df["L3"] = zip(*df["pathlist_names"].map(split_3levels))

display(df[["pathlist_names", "L1", "L2", "L3"]].head(10))
print("Unique L3:", df["L3"].nunique())

In [ ]:
df = df[df["L3"].notna()].copy()
df["L3"] = df["L3"].astype(str)

print("Filtered shape:", df.shape)
print("Unique L3 after filtering:", df["L3"].nunique())

In [ ]:
LEAK_COLS = [
    "pathlist_names",
    "pathlist_ids",
    "Category.Name.Value",
    "Category.CategoryID",
    "VirtualCategory",
    "L1",
    "L2",
    "L3"
]

FEATURE_CANDIDATES = [
    "Title",
    "Brand",
    "BrandInfo.BrandName",
    "ProductName",
    "BrandPartCode",
    "ProductFamily.Value",
    "ProductSeries.Value",
    "SummaryDescription.ShortSummaryDescription",
    "SummaryDescription.LongSummaryDescription",
    "Description.LongProductName",
    "Description.MiddleDesc",
    "Description.LongDesc",
    "BulletPoints.Values",
    "BulletPoints"
]

feature_cols = [c for c in FEATURE_CANDIDATES if c in df.columns]

print("Safe feature columns found:")
for col in feature_cols:
    print("-", col)

In [ ]:
X = df[feature_cols].copy()
y_true = df["L3"].copy()

print("X shape:", X.shape)
print("y_true shape:", y_true.shape)
display(X.head(3))

In [ ]:
X.to_parquet("/kaggle/working/icecat_features.parquet", index=False)
y_true.to_frame(name="L3").to_parquet("/kaggle/working/icecat_labels_l3.parquet", index=False)

print("Saved:")
print("/kaggle/working/icecat_features.parquet")
print("/kaggle/working/icecat_labels_l3.parquet")

Cell 10 — build clean product documents with spec tokens

In [ ]:
import re
import pandas as pd
import numpy as np

def clean_text(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    s = str(x)
    s = re.sub(r"<[^>]+>", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    if len(s) > 1200:
        s = s[:1200]
    return s

def listish_to_text(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ""
    if isinstance(v, list):
        out = []
        for item in v:
            if isinstance(item, dict):
                for k in ("Value", "Text", "value", "text"):
                    if k in item and item[k]:
                        out.append(clean_text(item[k]))
                        break
            else:
                t = clean_text(item)
                if t:
                    out.append(t)
        return " | ".join(out)
    if isinstance(v, dict):
        out = []
        for k in v:
            t = clean_text(v.get(k))
            if t:
                out.append(t)
        return " | ".join(out)
    return clean_text(v)

def extract_spec_tokens(text: str) -> str:
    if not text:
        return ""
    t = text.lower()
    out = []

    out += [f"gb:{m}" for m in re.findall(r"\b(\d{2,4})\s*gb\b", t)]
    out += [f"tb:{m}" for m in re.findall(r"\b(\d{1,2})\s*tb\b", t)]
    out += [f"mah:{m}" for m in re.findall(r"\b(\d{3,5})\s*mah\b", t)]
    out += [f"hz:{m}" for m in re.findall(r"\b(\d{2,4})\s*hz\b", t)]
    out += [f"w:{m}" for m in re.findall(r"\b(\d{2,4})\s*w\b", t)]
    out += [f"in:{m}" for m in re.findall(r'(\d{1,2}(?:\.\d)?)\s*(?:inch|inches|["])', t)]
    out += [f"res:{a}x{b}" for a, b in re.findall(r"\b(\d{3,4})\s*x\s*(\d{3,4})\b", t)]

    return " ".join(out[:50])

def row_to_doc(row):
    ordered_cols = [
        "Brand",
        "BrandInfo.BrandName",
        "Title",
        "ProductName",
        "BrandPartCode",
        "ProductFamily.Value",
        "ProductSeries.Value",
        "SummaryDescription.ShortSummaryDescription",
        "SummaryDescription.LongSummaryDescription",
        "Description.LongProductName",
        "Description.MiddleDesc",
        "Description.LongDesc",
        "BulletPoints.Values",
        "BulletPoints"
    ]

    parts = []
    for col in ordered_cols:
        if col not in row:
            continue

        v = row[col]
        if col in ("BulletPoints.Values", "BulletPoints"):
            v = listish_to_text(v)
        else:
            v = clean_text(v)

        if v:
            parts.append(f"{col.lower()}: {v}")

    doc = "\n".join(parts)
    spec = extract_spec_tokens(doc)
    if spec:
        doc = doc + "\n" + "spec_tokens: " + spec

    return doc

create the product documents

In [ ]:
from tqdm.auto import tqdm

tqdm.pandas()

docs = X.progress_apply(lambda row: row_to_doc(row.to_dict()), axis=1)

print("Number of documents:", len(docs))
print("\nSample document:\n")
print(docs.iloc[0][:2000])

In [ ]:
!pip -q install -U sentence-transformers

load docs and create embeddings

In [ ]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

docs_list = docs.tolist()

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model = SentenceTransformer("BAAI/bge-m3", device=device)
model.max_seq_length = 256

emb = model.encode(
    docs_list,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

emb = np.asarray(emb, dtype=np.float32)
print("Embeddings shape:", emb.shape)

load embeddings and verify normalization

In [ ]:
try:
    print("emb exists in memory:", emb.shape, emb.dtype)
except NameError:
    print("emb is NOT in memory")

In [ ]:
import numpy as np

np.save("/kaggle/working/icecat_bge_m3_embeddings.npy", emb)
print("Saved:", "/kaggle/working/icecat_bge_m3_embeddings.npy")

In [ ]:
import os
print(os.path.exists("/kaggle/working/icecat_bge_m3_embeddings.npy"))

In [ ]:
import numpy as np

emb = np.load("/kaggle/working/icecat_bge_m3_embeddings.npy")
print("Embeddings shape:", emb.shape)
print("dtype:", emb.dtype)
print("First vector norm:", np.linalg.norm(emb[0]))

In [ ]:
from sklearn.cluster import Birch
import numpy as np

birch_threshold = 0.22

birch = Birch(
    threshold=birch_threshold,
    n_clusters=None
)

micro_labels = birch.fit_predict(emb).astype(np.int32)

print("Micro labels shape:", micro_labels.shape)
print("Number of microclusters:", len(np.unique(micro_labels)))

In [ ]:
from sklearn.cluster import Birch
import numpy as np

birch_threshold = 0.30

birch = Birch(
    threshold=birch_threshold,
    n_clusters=None
)

micro_labels = birch.fit_predict(emb).astype(np.int32)

print("Micro labels shape:", micro_labels.shape)
print("Number of microclusters:", len(np.unique(micro_labels)))

In [ ]:
import pandas as pd
import numpy as np
import re
from tqdm.auto import tqdm

JSON_PATH = "/kaggle/input/datasets/bharatgangolli/icecat/icecat_data_train (1).json"

df = pd.read_json(JSON_PATH)

def split_3levels(x):
    if pd.isna(x):
        return None, None, None
    parts = [p.strip() for p in str(x).split(">")]
    parts += [None] * (3 - len(parts))
    return parts[0], parts[1], parts[2]

df["L1"], df["L2"], df["L3"] = zip(*df["pathlist_names"].map(split_3levels))
df = df[df["L3"].notna()].copy()
df["L3"] = df["L3"].astype(str)

FEATURE_CANDIDATES = [
    "Title",
    "Brand",
    "BrandInfo.BrandName",
    "ProductName",
    "BrandPartCode",
    "ProductFamily.Value",
    "ProductSeries.Value",
    "SummaryDescription.ShortSummaryDescription",
    "SummaryDescription.LongSummaryDescription",
    "Description.LongProductName",
    "Description.MiddleDesc",
    "Description.LongDesc",
    "BulletPoints.Values",
    "BulletPoints"
]

feature_cols = [c for c in FEATURE_CANDIDATES if c in df.columns]
X = df[feature_cols].copy()
y_true = df["L3"].copy()

def clean_text(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    s = str(x)
    s = re.sub(r"<[^>]+>", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    if len(s) > 1200:
        s = s[:1200]
    return s

def listish_to_text(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ""
    if isinstance(v, list):
        out = []
        for item in v:
            if isinstance(item, dict):
                for k in ("Value", "Text", "value", "text"):
                    if k in item and item[k]:
                        out.append(clean_text(item[k]))
                        break
            else:
                t = clean_text(item)
                if t:
                    out.append(t)
        return " | ".join(out)
    if isinstance(v, dict):
        out = []
        for k in v:
            t = clean_text(v.get(k))
            if t:
                out.append(t)
        return " | ".join(out)
    return clean_text(v)

def extract_spec_tokens(text: str) -> str:
    if not text:
        return ""
    t = text.lower()
    out = []

    out += [f"gb:{m}" for m in re.findall(r"\b(\d{2,4})\s*gb\b", t)]
    out += [f"tb:{m}" for m in re.findall(r"\b(\d{1,2})\s*tb\b", t)]
    out += [f"mah:{m}" for m in re.findall(r"\b(\d{3,5})\s*mah\b", t)]
    out += [f"hz:{m}" for m in re.findall(r"\b(\d{2,4})\s*hz\b", t)]
    out += [f"w:{m}" for m in re.findall(r"\b(\d{2,4})\s*w\b", t)]
    out += [f"in:{m}" for m in re.findall(r'(\d{1,2}(?:\.\d)?)\s*(?:inch|inches|["])', t)]
    out += [f"res:{a}x{b}" for a, b in re.findall(r"\b(\d{3,4})\s*x\s*(\d{3,4})\b", t)]

    return " ".join(out[:50])

def row_to_doc(row):
    ordered_cols = [
        "Brand",
        "BrandInfo.BrandName",
        "Title",
        "ProductName",
        "BrandPartCode",
        "ProductFamily.Value",
        "ProductSeries.Value",
        "SummaryDescription.ShortSummaryDescription",
        "SummaryDescription.LongSummaryDescription",
        "Description.LongProductName",
        "Description.MiddleDesc",
        "Description.LongDesc",
        "BulletPoints.Values",
        "BulletPoints"
    ]

    parts = []
    for col in ordered_cols:
        if col not in row:
            continue
        v = row[col]
        if col in ("BulletPoints.Values", "BulletPoints"):
            v = listish_to_text(v)
        else:
            v = clean_text(v)
        if v:
            parts.append(f"{col.lower()}: {v}")

    doc = "\n".join(parts)
    spec = extract_spec_tokens(doc)
    if spec:
        doc = doc + "\n" + "spec_tokens: " + spec
    return doc

tqdm.pandas()
docs = X.progress_apply(lambda row: row_to_doc(row.to_dict()), axis=1)

pd.DataFrame({"doc": docs}).to_parquet("/kaggle/working/icecat_docs.parquet", index=False)
y_true.to_frame(name="L3").to_parquet("/kaggle/working/icecat_labels_l3.parquet", index=False)

print("Done")
print("Rows:", len(docs))
print("Unique L3:", y_true.nunique())
print("Saved /kaggle/working/icecat_docs.parquet")
print("Saved /kaggle/working/icecat_labels_l3.parquet")
print("\nSample doc:\n")
print(docs.iloc[0][:1500])

Cell 2 — embeddings + save

In [ ]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
import pandas as pd

docs = pd.read_parquet("/kaggle/working/icecat_docs.parquet")["doc"].tolist()

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model = SentenceTransformer("BAAI/bge-m3", device=device)
model.max_seq_length = 256

emb = model.encode(
    docs,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

emb = np.asarray(emb, dtype=np.float32)

np.save("/kaggle/working/icecat_bge_m3_embeddings.npy", emb)

print("Saved: /kaggle/working/icecat_bge_m3_embeddings.npy")
print("Shape:", emb.shape)
print("First norm:", np.linalg.norm(emb[0]))

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
import numpy as np

emb = np.load("/kaggle/working/icecat_bge_m3_embeddings.npy")

pca = PCA(n_components=256, random_state=42)
emb_pca = pca.fit_transform(emb)
emb_pca = normalize(emb_pca, norm="l2").astype(np.float32)

np.save("/kaggle/working/icecat_bge_m3_pca256.npy", emb_pca)

print("Saved: /kaggle/working/icecat_bge_m3_pca256.npy")
print("PCA shape:", emb_pca.shape)
print("Explained variance ratio sum:", float(pca.explained_variance_ratio_.sum()))
print("First PCA norm:", np.linalg.norm(emb_pca[0]))

In [ ]:
import numpy as np
from sklearn.cluster import Birch

emb_pca = np.load("/kaggle/working/icecat_bge_m3_pca256.npy")

rng = np.random.RandomState(42)
sample_idx = rng.choice(len(emb_pca), size=100000, replace=False)
emb_sample = emb_pca[sample_idx]

thresholds = [0.6, 0.8, 1.0, 1.2]

for thr in thresholds:
    birch = Birch(threshold=thr, n_clusters=None)
    sample_labels = birch.fit_predict(emb_sample).astype(np.int32)
    n_micro = len(np.unique(sample_labels))
    print(f"threshold={thr} -> sample microclusters={n_micro}")

In [ ]:
import numpy as np
from sklearn.cluster import Birch

emb_pca = np.load("/kaggle/working/icecat_bge_m3_pca256.npy")

rng = np.random.RandomState(42)
sample_idx = rng.choice(len(emb_pca), size=50000, replace=False)
emb_sample = emb_pca[sample_idx]

thresholds = [0.70, 0.75]

for thr in thresholds:
    birch = Birch(threshold=thr, n_clusters=None)
    sample_labels = birch.fit_predict(emb_sample).astype(np.int32)
    n_micro = len(np.unique(sample_labels))
    print(f"threshold={thr} -> sample microclusters={n_micro}")

In [ ]:
import numpy as np
from sklearn.cluster import Birch

emb_pca = np.load("/kaggle/working/icecat_bge_m3_pca256.npy")

birch_threshold = 0.70

birch = Birch(
    threshold=birch_threshold,
    n_clusters=None
)

micro_labels = birch.fit_predict(emb_pca).astype(np.int32)

print("Micro labels shape:", micro_labels.shape)
print("Number of microclusters:", len(np.unique(micro_labels)))

In [ ]:
import numpy as np

micro_ids, inv = np.unique(micro_labels, return_inverse=True)
m = len(micro_ids)
d = emb_pca.shape[1]

centroids = np.zeros((m, d), dtype=np.float32)
counts = np.zeros(m, dtype=np.int32)

for i in range(len(inv)):
    c = inv[i]
    centroids[c] += emb_pca[i]
    counts[c] += 1

centroids /= counts[:, None]

centroid_norms = np.linalg.norm(centroids, axis=1, keepdims=True) + 1e-12
centroids = (centroids / centroid_norms).astype(np.float32)

print("Centroids shape:", centroids.shape)
print("Min cluster size:", counts.min())
print("Median cluster size:", np.median(counts))
print("Max cluster size:", counts.max())

In [ ]:
np.save("/kaggle/working/icecat_micro_labels_pca.npy", micro_labels)
np.save("/kaggle/working/icecat_micro_ids_pca.npy", micro_ids.astype(np.int32))
np.save("/kaggle/working/icecat_micro_centroids_pca.npy", centroids)
np.save("/kaggle/working/icecat_micro_counts_pca.npy", counts)

print("Saved:")
print("/kaggle/working/icecat_micro_labels_pca.npy")
print("/kaggle/working/icecat_micro_ids_pca.npy")
print("/kaggle/working/icecat_micro_centroids_pca.npy")
print("/kaggle/working/icecat_micro_counts_pca.npy")

In [ ]:
!pip -q install -U faiss-cpu igraph leidenalg

In [ ]:
import numpy as np
import faiss

centroids = np.load("/kaggle/working/icecat_micro_centroids_pca.npy")
print("Centroids shape:", centroids.shape)

m, d = centroids.shape
k = 50

index = faiss.IndexFlatIP(d)
index.add(centroids)

sims, nbrs = index.search(centroids, k)

print("sims shape:", sims.shape)
print("nbrs shape:", nbrs.shape)
print("Sample neighbor sims:", sims[0][:10])

In [ ]:
flat_sims = sims[:, 1:].ravel()   # exclude self-neighbor at column 0

for q in [0.80, 0.85, 0.90, 0.92, 0.95]:
    print(f"quantile {q}: {np.quantile(flat_sims, q):.4f}")

Cell 11 — build the Leiden graph

In [ ]:
import igraph as ig

sim_thr = float(np.quantile(flat_sims, 0.90))
sim_thr = max(0.25, min(sim_thr, 0.45))

print("Using similarity threshold:", sim_thr)

src_list = []
dst_list = []
w_list = []

for i in range(m):
    js = nbrs[i]
    ss = sims[i]

    mask = (js >= 0) & (js != i) & (ss >= sim_thr)
    js = js[mask].astype(np.int32)
    ss = ss[mask].astype(np.float32)

    if js.size:
        src_list.append(np.full(js.shape, i, dtype=np.int32))
        dst_list.append(js)
        w_list.append(ss)

src = np.concatenate(src_list)
dst = np.concatenate(dst_list)
wts = np.concatenate(w_list)

print("Edges kept:", len(wts))

g = ig.Graph(n=m, edges=list(zip(src.tolist(), dst.tolist())), directed=False)
g.es["weight"] = wts.tolist()

print(g.summary())

In [ ]:
import leidenalg
from sklearn.metrics import adjusted_mutual_info_score

def run_leiden(res, seed=42):
    part = leidenalg.find_partition(
        g,
        leidenalg.RBConfigurationVertexPartition,
        weights=g.es["weight"],
        resolution_parameter=float(res),
        seed=int(seed),
    )
    lab = np.array(part.membership, dtype=np.int32)
    uniq, cnts = np.unique(lab, return_counts=True)
    singleton_ratio = float((cnts == 1).mean())
    return lab, len(uniq), singleton_ratio

res_grid = [2.0, 4.0, 6.0, 8.0, 10.0, 12.0]
best = None

for res in res_grid:
    lab1, k1, s1 = run_leiden(res, seed=1)
    lab2, k2, s2 = run_leiden(res, seed=2)

    stab = adjusted_mutual_info_score(lab1, lab2)
    score = stab - 0.7 * s1

    print(f"res={res:.1f}  clusters={k1}  singletons={s1:.4f}  stability={stab:.4f}  score={score:.4f}")

    if best is None or score > best["score"]:
        best = {"res": res, "score": score}

print("\nChosen resolution:", best["res"])

In [ ]:
best_res = best["res"]

part = leidenalg.find_partition(
    g,
    leidenalg.RBConfigurationVertexPartition,
    weights=g.es["weight"],
    resolution_parameter=float(best_res),
    seed=42,
)

centroid_cluster = np.array(part.membership, dtype=np.int32)

print("Final centroid clusters:", len(np.unique(centroid_cluster)))

In [ ]:
micro_labels = np.load("/kaggle/working/icecat_micro_labels_pca.npy")
micro_ids = np.load("/kaggle/working/icecat_micro_ids_pca.npy")

micro_idx = np.searchsorted(micro_ids, micro_labels)
final_clusters = centroid_cluster[micro_idx].astype(np.int32)

print("Final product cluster labels shape:", final_clusters.shape)
print("Number of final clusters:", len(np.unique(final_clusters)))

np.save("/kaggle/working/icecat_final_clusters_purity.npy", final_clusters)
print("Saved: /kaggle/working/icecat_final_clusters_purity.npy")

In [ ]:
import pandas as pd
from collections import Counter
from sklearn.metrics import (
    normalized_mutual_info_score,
    adjusted_rand_score,
    v_measure_score,
    adjusted_mutual_info_score,
)

y_true = pd.read_parquet("/kaggle/working/icecat_labels_l3.parquet")["L3"].astype(str).to_numpy()
y_pred = np.load("/kaggle/working/icecat_final_clusters_purity.npy")

def purity_score(y_true, y_pred):
    total = len(y_true)
    s = 0
    for c in np.unique(y_pred):
        idx = np.where(y_pred == c)[0]
        most = Counter(y_true[idx]).most_common(1)[0][1]
        s += most
    return s / total

metrics = {
    "purity": purity_score(y_true, y_pred),
    "nmi": normalized_mutual_info_score(y_true, y_pred),
    "ari": adjusted_rand_score(y_true, y_pred),
    "v_measure": v_measure_score(y_true, y_pred),
    "ami": adjusted_mutual_info_score(y_true, y_pred),
    "n_clusters": len(np.unique(y_pred)),
}

print(metrics)

In [ ]:
import numpy as np
import faiss
import igraph as ig
import leidenalg
import pandas as pd

from collections import Counter
from sklearn.metrics import (
    normalized_mutual_info_score,
    adjusted_rand_score,
    v_measure_score,
    adjusted_mutual_info_score,
)

centroids = np.load("/kaggle/working/icecat_micro_centroids_pca.npy")
micro_labels = np.load("/kaggle/working/icecat_micro_labels_pca.npy")
micro_ids = np.load("/kaggle/working/icecat_micro_ids_pca.npy")
y_true = pd.read_parquet("/kaggle/working/icecat_labels_l3.parquet")["L3"].astype(str).to_numpy()

m, d = centroids.shape
k = 50

index = faiss.IndexFlatIP(d)
index.add(centroids)
sims, nbrs = index.search(centroids, k)

flat_sims = sims[:, 1:].ravel()

def purity_score(y_true, y_pred):
    total = len(y_true)
    s = 0
    for c in np.unique(y_pred):
        idx = np.where(y_pred == c)[0]
        most = Counter(y_true[idx]).most_common(1)[0][1]
        s += most
    return s / total

results = []

quantiles = [0.90, 0.92, 0.94, 0.95]
res_grid = [8, 12, 16, 20, 24, 30, 40]

for q in quantiles:
    sim_thr = float(np.quantile(flat_sims, q))
    sim_thr = max(0.25, min(sim_thr, 0.50))

    src_list = []
    dst_list = []
    w_list = []

    for i in range(m):
        js = nbrs[i]
        ss = sims[i]

        mask = (js >= 0) & (js != i) & (ss >= sim_thr)
        js = js[mask].astype(np.int32)
        ss = ss[mask].astype(np.float32)

        if js.size:
            src_list.append(np.full(js.shape, i, dtype=np.int32))
            dst_list.append(js)
            w_list.append(ss)

    src = np.concatenate(src_list)
    dst = np.concatenate(dst_list)
    wts = np.concatenate(w_list)

    g = ig.Graph(n=m, edges=list(zip(src.tolist(), dst.tolist())), directed=False)
    g.es["weight"] = wts.tolist()

    for res in res_grid:
        part = leidenalg.find_partition(
            g,
            leidenalg.RBConfigurationVertexPartition,
            weights=g.es["weight"],
            resolution_parameter=float(res),
            seed=42,
        )

        centroid_cluster = np.array(part.membership, dtype=np.int32)

        micro_idx = np.searchsorted(micro_ids, micro_labels)
        final_clusters = centroid_cluster[micro_idx].astype(np.int32)

        metrics = {
            "quantile": q,
            "sim_thr": sim_thr,
            "resolution": res,
            "purity": purity_score(y_true, final_clusters),
            "nmi": normalized_mutual_info_score(y_true, final_clusters),
            "ari": adjusted_rand_score(y_true, final_clusters),
            "v_measure": v_measure_score(y_true, final_clusters),
            "ami": adjusted_mutual_info_score(y_true, final_clusters),
            "n_clusters": len(np.unique(final_clusters)),
        }
        results.append(metrics)
        print(metrics)

results_df = pd.DataFrame(results).sort_values(
    by=["purity", "ami", "ari"],
    ascending=False
).reset_index(drop=True)

print("\nTop 15 results:")
display(results_df.head(15))

results_df.to_csv("/kaggle/working/leiden_sweep_results.csv", index=False)
print("Saved: /kaggle/working/leiden_sweep_results.csv")

In [ ]:
import numpy as np
from sklearn.cluster import Birch

emb_pca = np.load("/kaggle/working/icecat_bge_m3_pca256.npy")

birch_threshold = 0.60

birch = Birch(
    threshold=birch_threshold,
    n_clusters=None
)

micro_labels = birch.fit_predict(emb_pca).astype(np.int32)

print("Micro labels shape:", micro_labels.shape)
print("Number of microclusters:", len(np.unique(micro_labels)))

In [ ]:
import numpy as np

micro_ids, inv = np.unique(micro_labels, return_inverse=True)
m = len(micro_ids)
d = emb_pca.shape[1]

centroids = np.zeros((m, d), dtype=np.float32)
counts = np.zeros(m, dtype=np.int32)

for i in range(len(inv)):
    c = inv[i]
    centroids[c] += emb_pca[i]
    counts[c] += 1

centroids /= counts[:, None]

centroid_norms = np.linalg.norm(centroids, axis=1, keepdims=True) + 1e-12
centroids = (centroids / centroid_norms).astype(np.float32)

print("Centroids shape:", centroids.shape)
print("Min cluster size:", counts.min())
print("Median cluster size:", np.median(counts))
print("Max cluster size:", counts.max())

In [ ]:
np.save("/kaggle/working/icecat_micro_labels_pca_t060.npy", micro_labels)
np.save("/kaggle/working/icecat_micro_ids_pca_t060.npy", micro_ids.astype(np.int32))
np.save("/kaggle/working/icecat_micro_centroids_pca_t060.npy", centroids)
np.save("/kaggle/working/icecat_micro_counts_pca_t060.npy", counts)

print("Saved new t=0.60 BIRCH artifacts")

In [ ]:
import numpy as np
import faiss
import igraph as ig
import leidenalg
import pandas as pd

from collections import Counter
from sklearn.metrics import (
    normalized_mutual_info_score,
    adjusted_rand_score,
    v_measure_score,
    adjusted_mutual_info_score,
)

centroids = np.load("/kaggle/working/icecat_micro_centroids_pca_t060.npy")
micro_labels = np.load("/kaggle/working/icecat_micro_labels_pca_t060.npy")
micro_ids = np.load("/kaggle/working/icecat_micro_ids_pca_t060.npy")
y_true = pd.read_parquet("/kaggle/working/icecat_labels_l3.parquet")["L3"].astype(str).to_numpy()

m, d = centroids.shape
k = 80

index = faiss.IndexFlatIP(d)
index.add(centroids)
sims, nbrs = index.search(centroids, k)

flat_sims = sims[:, 1:].ravel()

print("Similarity quantiles:")
for q in [0.90, 0.95, 0.97, 0.98, 0.99]:
    print(q, float(np.quantile(flat_sims, q)))

def purity_score(y_true, y_pred):
    total = len(y_true)
    s = 0
    for c in np.unique(y_pred):
        idx = np.where(y_pred == c)[0]
        most = Counter(y_true[idx]).most_common(1)[0][1]
        s += most
    return s / total

results = []

quantiles = [0.95, 0.97, 0.98, 0.99]
res_grid = [20, 30, 40, 50, 60, 80]

for q in quantiles:
    sim_thr = float(np.quantile(flat_sims, q))
    sim_thr = max(0.25, sim_thr)

    src_list = []
    dst_list = []
    w_list = []

    for i in range(m):
        js = nbrs[i]
        ss = sims[i]

        mask = (js >= 0) & (js != i) & (ss >= sim_thr)
        js = js[mask].astype(np.int32)
        ss = ss[mask].astype(np.float32)

        if js.size:
            src_list.append(np.full(js.shape, i, dtype=np.int32))
            dst_list.append(js)
            w_list.append(ss)

    src = np.concatenate(src_list)
    dst = np.concatenate(dst_list)
    wts = np.concatenate(w_list)

    print(f"\nquantile={q} sim_thr={sim_thr:.4f} edges={len(wts)}")

    g = ig.Graph(n=m, edges=list(zip(src.tolist(), dst.tolist())), directed=False)
    g.es["weight"] = wts.tolist()

    for res in res_grid:
        part = leidenalg.find_partition(
            g,
            leidenalg.RBConfigurationVertexPartition,
            weights=g.es["weight"],
            resolution_parameter=float(res),
            seed=42,
        )

        centroid_cluster = np.array(part.membership, dtype=np.int32)
        micro_idx = np.searchsorted(micro_ids, micro_labels)
        final_clusters = centroid_cluster[micro_idx].astype(np.int32)

        metrics = {
            "quantile": q,
            "sim_thr": sim_thr,
            "resolution": res,
            "purity": purity_score(y_true, final_clusters),
            "nmi": normalized_mutual_info_score(y_true, final_clusters),
            "ari": adjusted_rand_score(y_true, final_clusters),
            "v_measure": v_measure_score(y_true, final_clusters),
            "ami": adjusted_mutual_info_score(y_true, final_clusters),
            "n_clusters": len(np.unique(final_clusters)),
        }
        results.append(metrics)
        print(metrics)

results_df = pd.DataFrame(results).sort_values(
    by=["purity", "ami", "ari"],
    ascending=False
).reset_index(drop=True)

print("\nTop 15 results:")
display(results_df.head(15))

results_df.to_csv("/kaggle/working/leiden_sweep_results_t060.csv", index=False)
print("Saved: /kaggle/working/leiden_sweep_results_t060.csv")

In [ ]:
import numpy as np
import faiss
import igraph as ig
import leidenalg
import pandas as pd

centroids = np.load("/kaggle/working/icecat_micro_centroids_pca_t060.npy")
micro_labels = np.load("/kaggle/working/icecat_micro_labels_pca_t060.npy")
micro_ids = np.load("/kaggle/working/icecat_micro_ids_pca_t060.npy")

m, d = centroids.shape
k = 80

index = faiss.IndexFlatIP(d)
index.add(centroids)
sims, nbrs = index.search(centroids, k)

flat_sims = sims[:, 1:].ravel()
sim_thr = float(np.quantile(flat_sims, 0.99))

src_list = []
dst_list = []
w_list = []

for i in range(m):
    js = nbrs[i]
    ss = sims[i]

    mask = (js >= 0) & (js != i) & (ss >= sim_thr)
    js = js[mask].astype(np.int32)
    ss = ss[mask].astype(np.float32)

    if js.size:
        src_list.append(np.full(js.shape, i, dtype=np.int32))
        dst_list.append(js)
        w_list.append(ss)

src = np.concatenate(src_list)
dst = np.concatenate(dst_list)
wts = np.concatenate(w_list)

g = ig.Graph(n=m, edges=list(zip(src.tolist(), dst.tolist())), directed=False)
g.es["weight"] = wts.tolist()

part = leidenalg.find_partition(
    g,
    leidenalg.RBConfigurationVertexPartition,
    weights=g.es["weight"],
    resolution_parameter=60.0,
    seed=42,
)

centroid_cluster = np.array(part.membership, dtype=np.int32)
micro_idx = np.searchsorted(micro_ids, micro_labels)
final_clusters_purity = centroid_cluster[micro_idx].astype(np.int32)

np.save("/kaggle/working/icecat_final_clusters_purity_95.npy", final_clusters_purity)
print("Saved: /kaggle/working/icecat_final_clusters_purity_95.npy")
print("Clusters:", len(np.unique(final_clusters_purity)))

In [ ]:
import numpy as np

emb_pca = np.load("/kaggle/working/icecat_bge_m3_pca256.npy")
fine_labels = np.load("/kaggle/working/icecat_final_clusters_purity_95.npy").astype(np.int32)

fine_ids, fine_inv = np.unique(fine_labels, return_inverse=True)
k0 = len(fine_ids)
d = emb_pca.shape[1]

fine_centroids = np.zeros((k0, d), dtype=np.float32)
fine_counts = np.zeros(k0, dtype=np.int32)

for i in range(len(fine_inv)):
    c = fine_inv[i]
    fine_centroids[c] += emb_pca[i]
    fine_counts[c] += 1

fine_centroids /= fine_counts[:, None]
fine_centroids /= (np.linalg.norm(fine_centroids, axis=1, keepdims=True) + 1e-12)

print("Fine centroids shape:", fine_centroids.shape)
print("Min size:", fine_counts.min())
print("Median size:", np.median(fine_counts))
print("Max size:", fine_counts.max())

In [ ]:
import faiss
import igraph as ig
import leidenalg
import pandas as pd
from collections import Counter
from sklearn.metrics import (
    normalized_mutual_info_score,
    adjusted_rand_score,
    v_measure_score,
    adjusted_mutual_info_score,
)

y_true = pd.read_parquet("/kaggle/working/icecat_labels_l3.parquet")["L3"].astype(str).to_numpy()

index = faiss.IndexFlatIP(fine_centroids.shape[1])
index.add(fine_centroids)

k_merge = 30
sims2, nbrs2 = index.search(fine_centroids, k_merge)

def purity_score(y_true, y_pred):
    total = len(y_true)
    s = 0
    for c in np.unique(y_pred):
        idx = np.where(y_pred == c)[0]
        most = Counter(y_true[idx]).most_common(1)[0][1]
        s += most
    return s / total

merge_thresholds = [0.90, 0.92, 0.94, 0.96, 0.97, 0.98]
merge_results = []

for thr in merge_thresholds:
    src_list, dst_list, w_list = [], [], []

    for i in range(k0):
        js = nbrs2[i]
        ss = sims2[i]

        mask = (js >= 0) & (js != i) & (ss >= thr)
        js = js[mask].astype(np.int32)
        ss = ss[mask].astype(np.float32)

        if js.size:
            src_list.append(np.full(js.shape, i, dtype=np.int32))
            dst_list.append(js)
            w_list.append(ss)

    if not src_list:
        print(f"thr={thr} -> no edges")
        continue

    src = np.concatenate(src_list)
    dst = np.concatenate(dst_list)
    wts = np.concatenate(w_list)

    g2 = ig.Graph(n=k0, edges=list(zip(src.tolist(), dst.tolist())), directed=False)
    g2.es["weight"] = wts.tolist()

    part2 = leidenalg.find_partition(
        g2,
        leidenalg.RBConfigurationVertexPartition,
        weights=g2.es["weight"],
        resolution_parameter=0.5,
        seed=42,
    )

    merged_centroid_labels = np.array(part2.membership, dtype=np.int32)
    balanced_clusters = merged_centroid_labels[fine_inv].astype(np.int32)

    metrics = {
        "merge_thr": thr,
        "purity": purity_score(y_true, balanced_clusters),
        "nmi": normalized_mutual_info_score(y_true, balanced_clusters),
        "ari": adjusted_rand_score(y_true, balanced_clusters),
        "v_measure": v_measure_score(y_true, balanced_clusters),
        "ami": adjusted_mutual_info_score(y_true, balanced_clusters),
        "n_clusters": len(np.unique(balanced_clusters)),
    }
    merge_results.append(metrics)
    print(metrics)

merge_df = pd.DataFrame(merge_results).sort_values(
    by=["ari", "ami", "nmi"],
    ascending=False
).reset_index(drop=True)

print("\nMerge results sorted by structural similarity:")
display(merge_df)

In [ ]:
import numpy as np
import faiss
import igraph as ig
import leidenalg
import pandas as pd
from collections import Counter
from sklearn.metrics import (
    normalized_mutual_info_score,
    adjusted_rand_score,
    v_measure_score,
    adjusted_mutual_info_score,
)

y_true = pd.read_parquet("/kaggle/working/icecat_labels_l3.parquet")["L3"].astype(str).to_numpy()

index = faiss.IndexFlatIP(fine_centroids.shape[1])
index.add(fine_centroids)

k_merge = 30
sims2, nbrs2 = index.search(fine_centroids, k_merge)

print("Merge similarity quantiles:")
flat2 = sims2[:, 1:].ravel()
for q in [0.80, 0.85, 0.90, 0.95, 0.97, 0.98, 0.99]:
    print(q, float(np.quantile(flat2, q)))

def purity_score(y_true, y_pred):
    total = len(y_true)
    s = 0
    for c in np.unique(y_pred):
        idx = np.where(y_pred == c)[0]
        most = Counter(y_true[idx]).most_common(1)[0][1]
        s += most
    return s / total

merge_thresholds = [0.70, 0.75, 0.80, 0.85, 0.90, 0.92, 0.94, 0.96]
merge_results = []

for thr in merge_thresholds:
    src_list, dst_list, w_list = [], [], []

    for i in range(k0):
        js = nbrs2[i]
        ss = sims2[i]

        mask = (js >= 0) & (js != i) & (ss >= thr)
        js = js[mask].astype(np.int32)
        ss = ss[mask].astype(np.float32)

        if js.size:
            src_list.append(np.full(js.shape, i, dtype=np.int32))
            dst_list.append(js)
            w_list.append(ss)

    if len(src_list) == 0:
        print(f"thr={thr:.2f} -> no edges, skipped")
        continue

    src = np.concatenate(src_list)
    dst = np.concatenate(dst_list)
    wts = np.concatenate(w_list)

    g2 = ig.Graph(n=k0, edges=list(zip(src.tolist(), dst.tolist())), directed=False)
    g2.es["weight"] = wts.tolist()

    part2 = leidenalg.find_partition(
        g2,
        leidenalg.RBConfigurationVertexPartition,
        weights=g2.es["weight"],
        resolution_parameter=0.5,
        seed=42,
    )

    merged_centroid_labels = np.array(part2.membership, dtype=np.int32)
    balanced_clusters = merged_centroid_labels[fine_inv].astype(np.int32)

    metrics = {
        "merge_thr": thr,
        "edges": int(len(wts)),
        "purity": purity_score(y_true, balanced_clusters),
        "nmi": normalized_mutual_info_score(y_true, balanced_clusters),
        "ari": adjusted_rand_score(y_true, balanced_clusters),
        "v_measure": v_measure_score(y_true, balanced_clusters),
        "ami": adjusted_mutual_info_score(y_true, balanced_clusters),
        "n_clusters": len(np.unique(balanced_clusters)),
    }
    merge_results.append(metrics)
    print(metrics)

merge_df = pd.DataFrame(merge_results).sort_values(
    by=["ari", "ami", "nmi"],
    ascending=False
).reset_index(drop=True)

print("\nMerge results sorted by structural similarity:")
display(merge_df)

In [ ]:
chosen_thr = 0.75
print("Chosen threshold:", chosen_thr)

src_list, dst_list, w_list = [], [], []

for i in range(k0):
    js = nbrs2[i]
    ss = sims2[i]

    mask = (js >= 0) & (js != i) & (ss >= chosen_thr)
    js = js[mask].astype(np.int32)
    ss = ss[mask].astype(np.float32)

    if js.size:
        src_list.append(np.full(js.shape, i, dtype=np.int32))
        dst_list.append(js)
        w_list.append(ss)

src = np.concatenate(src_list)
dst = np.concatenate(dst_list)
wts = np.concatenate(w_list)

g2 = ig.Graph(n=k0, edges=list(zip(src.tolist(), dst.tolist())), directed=False)
g2.es["weight"] = wts.tolist()

part2 = leidenalg.find_partition(
    g2,
    leidenalg.RBConfigurationVertexPartition,
    weights=g2.es["weight"],
    resolution_parameter=0.5,
    seed=42,
)

merged_centroid_labels = np.array(part2.membership, dtype=np.int32)
balanced_clusters = merged_centroid_labels[fine_inv].astype(np.int32)

np.save("/kaggle/working/icecat_final_clusters_balanced.npy", balanced_clusters)

print("Saved: /kaggle/working/icecat_final_clusters_balanced.npy")
print("Balanced clusters:", len(np.unique(balanced_clusters)))

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.metrics import (
    normalized_mutual_info_score,
    adjusted_rand_score,
    v_measure_score,
    adjusted_mutual_info_score,
)

y_true = pd.read_parquet("/kaggle/working/icecat_labels_l3.parquet")["L3"].astype(str).to_numpy()
purity_clusters = np.load("/kaggle/working/icecat_final_clusters_purity_95.npy")
balanced_clusters = np.load("/kaggle/working/icecat_final_clusters_balanced.npy")

def purity_score(y_true, y_pred):
    total = len(y_true)
    s = 0
    for c in np.unique(y_pred):
        idx = np.where(y_pred == c)[0]
        most = Counter(y_true[idx]).most_common(1)[0][1]
        s += most
    return s / total

rows = []
for name, y_pred in [
    ("purity_optimized", purity_clusters),
    ("balanced", balanced_clusters),
]:
    rows.append({
        "variant": name,
        "purity": purity_score(y_true, y_pred),
        "nmi": normalized_mutual_info_score(y_true, y_pred),
        "ari": adjusted_rand_score(y_true, y_pred),
        "v_measure": v_measure_score(y_true, y_pred),
        "ami": adjusted_mutual_info_score(y_true, y_pred),
        "n_clusters": len(np.unique(y_pred)),
    })

final_df = pd.DataFrame(rows)
display(final_df)

final_df.to_csv("/kaggle/working/final_metrics_comparison.csv", index=False)
print("Saved: /kaggle/working/final_metrics_comparison.csv")

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize

# load saved PCA embeddings and labels
emb_pca = np.load("/kaggle/working/icecat_bge_m3_pca256.npy")
final_clusters = np.load("/kaggle/working/icecat_final_clusters_balanced.npy")
y_true = pd.read_parquet("/kaggle/working/icecat_labels_l3.parquet")["L3"].astype(str).to_numpy()

# rebuild PCA object if not already in memory
emb = np.load("/kaggle/working/icecat_bge_m3_embeddings.npy")
pca = PCA(n_components=256, random_state=42)
emb_pca_check = pca.fit_transform(emb)
emb_pca_check = normalize(emb_pca_check, norm="l2").astype(np.float32)

# cluster centroids
cluster_ids, inv = np.unique(final_clusters, return_inverse=True)

centroids = np.zeros((len(cluster_ids), emb_pca.shape[1]), dtype=np.float32)
counts = np.zeros(len(cluster_ids), dtype=np.int32)

for i in range(len(inv)):
    c = inv[i]
    centroids[c] += emb_pca[i]
    counts[c] += 1

centroids /= counts[:, None]
centroids /= (np.linalg.norm(centroids, axis=1, keepdims=True) + 1e-12)

# majority L3 label per cluster
labels = []
for cid in cluster_ids:
    idx = np.where(final_clusters == cid)[0]
    maj = Counter(y_true[idx]).most_common(1)[0][0]
    labels.append(maj)

# save artifacts
joblib.dump(pca, "/kaggle/working/pca_256.joblib")
np.save("/kaggle/working/l3_centroids.npy", centroids.astype(np.float32))

with open("/kaggle/working/l3_labels.json", "w", encoding="utf-8") as f:
    json.dump(labels, f, ensure_ascii=False, indent=2)

print("Saved:")
print("/kaggle/working/pca_256.joblib")
print("/kaggle/working/l3_centroids.npy")
print("/kaggle/working/l3_labels.json")
print("Number of labels:", len(labels))
print("Centroids shape:", centroids.shape)

In [ ]:
import pandas as pd

JSON_PATH = "/kaggle/input/datasets/bharatgangolli/icecat/icecat_data_train (1).json"
df = pd.read_json(JSON_PATH)

demo_cols = [
    "Brand",
    "Title",
    "ProductName",
    "SummaryDescription.ShortSummaryDescription",
    "Description.LongDesc",
    "BulletPoints.Values",
]
demo_cols = [c for c in demo_cols if c in df.columns]

demo_df = df[demo_cols].dropna(subset=["Title"]).sample(20, random_state=42).reset_index(drop=True)

demo_path = "/kaggle/working/demo_products.csv"
demo_df.to_csv(demo_path, index=False)

print("Saved:", demo_path)
print("Shape:", demo_df.shape)
display(demo_df.head(10))